# Runtime


$\newcommand{\ket}[1]{\left|#1\right\rangle} \newcommand{\bra}[1]{\left\langle #1\right|} \newcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle} \newcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}
$This notebook expains how to calculate the runtime of each device. 

## Promise on the annealing schedule

We consider an annealing schedule $\beta_0=0<\beta_1<\cdots<\beta_F$ where $\beta_F$ reaches at most the value $100$. 

For each consecutive pair, we require
$$\left(\sum_x \sqrt{\pi_{\beta_k}(x)\pi_{\beta_{k+1}}(x)}\right)^2 = |\braket{\pi_{\beta_k}}{\pi_{\beta_{k+1}}}|^2 \ge 1/e$$

## Per-device runtime

For the classical local and uniform walks, the runtime is the number of executed Metropolis steps times a fitted per-step cost. For an annealing schedule with query counts $q_1,\ldots,q_T$, the total number of steps is $Q=\sum_{t=1}^T q_t$.

* Local proposal:
  * CPU: $T_{\rm CPU}^{\rm loc}(n)=Q\left(5.959\cdot10^{-9}+1.429\cdot10^{-10}n\right)$.
  * GPU: $T_{\rm GPU}^{\rm loc}(n)=Q\left(7.837\cdot10^{-7}+1.459\cdot10^{-9}n\right)$.
  * FPGA: $T_{\rm FPGA}^{\rm loc}(n)=Q\left(2.679\cdot10^{-7}+1.800\cdot10^{-9}\log_2 n\right)$.

* Uniform proposal:
  * CPU: $T_{\rm CPU}^{\rm unif}(n)=Q\left(1.173\cdot10^{-8}n+6.964\cdot10^{-11}n^2\right)$.
  * GPU: $T_{\rm GPU}^{\rm unif}(n)=Q\left(2.215\cdot10^{-10}n^2\right)$.
  * FPGA: $T_{\rm FPGA}^{\rm unif}(n)=Q\left(2.541\cdot10^{-7}+4.200\cdot10^{-9}\log_2 n\right)$.

For the classical walk with QEMC move, each Metropolis proposal calls a small quantum circuit. The surface-code distance is chosen from the spacetime volume of one Hamiltonian-simulation query, and the resulting physical time is multiplied by the number of queries.

* QEMC proposal:
  * Logical time per query: $L^{\rm qemc}(n)=1+r(n+2)$, with $r=50$ in the default setting.
  * Logical space per query: $S^{\rm qemc}(n)=n$.
  * Surface-code distance: $d=d_{\rm qemc}(n)$ is the smallest odd integer such that $L^{\rm qemc}(n)S^{\rm qemc}(n)d\,0.1(100p_{\rm phys})^{(d+1)/2}\le 1/3$.
  * Physical time: $T^{\rm qemc}(n)=Q\,L^{\rm qemc}(n)d_{\rm qemc}(n)(4t_{\rm phys}+t_{\rm meas})$.

The quantum walk approach can be implemented either by a sequence of QPE calls or by a sequence of spectral filters via QSP/QSVT. Since we do not need to keep an eigenvalue register, we use the QSP-filter implementation. At annealing step $t$, the filter degree is $f_t=2\lceil\ln(1/\varepsilon)/\Delta_{\theta,t}\rceil$, where $\Delta_{\theta,t}=\arccos(1-\delta_t)$ is the phase gap. We set $F=\sum_t f_t$ and use implementation error $\frac{\varepsilon}{F}$ inside each walk circuit.

* Quantum walk with local move:
  * Logical time for a single local walk: $L_Q^{\rm local}(n,\beta_t,\frac{\varepsilon}{F})$.
  * Logical space for a single local walk: $S_Q^{\rm local}(n,\frac{\varepsilon}{F})$.
  * Surface-code distance for the local move: $d_Q^{\rm local}(n)$ is chosen from the full-walk spacetime volume $V_Q^{\rm local}=\left(\sum_t f_tL_Q^{\rm local}(n,\beta_t,\frac{\varepsilon}{F})\right)S_Q^{\rm local}(n,\frac{\varepsilon}{F})$.
  * Quantum walk with local move: $T_Q^{\rm local}(n)=d_Q^{\rm local}(n)(4t_{\rm phys}+t_{\rm meas})\sum_t f_tL_Q^{\rm local}(n,\beta_t,\frac{\varepsilon}{F})$.

* Quantum walk with uniform move:
  * Logical time for a single uniform walk: $L_Q^{\rm uniform}(n,\beta_t,\frac{\varepsilon}{F})$.
  * Logical space for a single uniform walk: $S_Q^{\rm uniform}(n,\frac{\varepsilon}{F})$.
  * Surface-code distance for the uniform move: $d_Q^{\rm uniform}(n)$ is chosen from the full-walk spacetime volume $V_Q^{\rm uniform}=\left(\sum_t f_tL_Q^{\rm uniform}(n,\beta_t,\frac{\varepsilon}{F})\right)S_Q^{\rm uniform}(n,\frac{\varepsilon}{F})$.
  * Quantum walk with uniform move: $T_Q^{\rm uniform}(n)=d_Q^{\rm uniform}(n)(4t_{\rm phys}+t_{\rm meas})\sum_t f_tL_Q^{\rm uniform}(n,\beta_t,\frac{\varepsilon}{F})$.

* Quantum walk with QEMC move:
  * Logical time for a single QEMC walk: $L_Q^{\rm qemc}(n,\beta_t,\frac{\varepsilon}{F})$.
  * Logical space for a single QEMC walk: $S_Q^{\rm qemc}(n,\frac{\varepsilon}{F})$.
  * Surface-code distance for the QEMC move: $d_Q^{\rm qemc}(n)$ is chosen from the full-walk spacetime volume $V_Q^{\rm qemc}=\left(\sum_t f_tL_Q^{\rm qemc}(n,\beta_t,\frac{\varepsilon}{F})\right)S_Q^{\rm qemc}(n,\frac{\varepsilon}{F})$.
  * Quantum walk with QEMC move: $T_Q^{\rm qemc}(n)=d_Q^{\rm qemc}(n)(4t_{\rm phys}+t_{\rm meas})\sum_t f_tL_Q^{\rm qemc}(n,\beta_t,\frac{\varepsilon}{F})$.

## Worst case scenario on the annealing step

[Wocjan and Abeyesinghe, Section V](https://arxiv.org/pdf/0804.4259) gives a worst-case annealing-step rule for Boltzmann-Gibbs states. For a classical Hamiltonian $H$ with energies $E(x)\ge0$, the coherent Gibbs states satisfy 
$$|\braket{\pi_\beta}{\pi_{\beta+\Delta\beta}}|^2\ge \exp(-\|H\|\Delta\beta)$$ 
for all $\beta$ and all $\Delta\beta$. Here $\| H \|$ is the operator norm. 

Therefore, choosing $\Delta\beta=1/\|H\|$ guarantees $|\braket{\pi_\beta}{\pi_{\beta+\Delta\beta}}|^2 \ge 1/e$. In the worst case, reaching inverse temperature $\beta$ requires $r=\beta\|H\|$ annealing steps, with $\beta_i=i/\|H\|$ for $i=0,\ldots,r$.

For the SK model we have that $\|H\| \approx 2n$ (shown numerically in notebook 0_data.ipynb).